In [1]:
# Load the loop scores of every model and case (Loop_TEP-*-SetpointForecast.ipynb)
import json

import h5py
import numpy as np
import pandas as pd

CASES = {   # results file suffix: description
    "clean_sp": "No measurement noise, constant disturbances (tep_sp_spaced_6min_clean.h5), set points as covariates",
    "noisy_sp": "Measurement noise and AR(1) disturbances (tep_sp_spaced_6min_noisy.h5), set points as covariates",
    "noisy_pastdist": "Measurement noise and AR(1) disturbances (tep_sp_spaced_6min_noisy.h5), set points as covariates, "
                      "disturbances as past-only covariates (ARX cannot use these)",
    "constdist_sp": "Measurement noise, constant disturbances (tep_sp_spaced_6min_constdist_noisy.h5), set points as "
                    "covariates, errors scaled by the std of tep_sp_spaced_6min_noisy.h5",
    "ramp_sp": "Measurement noise, disturbances ramped between operating points (tep_sp_spaced_ramp_noisy.h5), set points "
               "as covariates, errors scaled by the std of tep_sp_spaced_6min_noisy.h5",
    "noisy_futuredist": "Measurement noise and AR(1) disturbances (tep_sp_spaced_6min_noisy.h5), set points and "
                        "disturbances as covariates, both known over the forecast",
    "constdist_futuredist": "Measurement noise, constant disturbances (tep_sp_spaced_6min_constdist_noisy.h5), set points "
                            "and disturbances known over the forecast, errors scaled by the std of tep_sp_spaced_6min_noisy.h5",
    "ramp_futuredist": "Measurement noise, ramped disturbances (tep_sp_spaced_ramp_noisy.h5), set points and disturbances "
                       "known over the forecast, errors scaled by the std of tep_sp_spaced_6min_noisy.h5",
}
MODELS = {"chronos2": "Chronos-2", "timesfm3": "TimesFM 3.0", "arx": "ARX + Ridge"}
METRICS = ["scaled MAE", "MAPE (%)", "coverage", "scaled Winkler"]

scores, per_variable, settings = {}, {}, {}
for case in CASES:
    for name, model in MODELS.items():
        with h5py.File(f"Results/Loop_setpointforecast_{name}_{case}.h5", "r") as f:
            settings[case, model] = json.loads(f.attrs["settings"])
            events = f["event"][()]
            scores[case, model] = pd.DataFrame({m: f[f"per_event/{m}"][()] for m in METRICS}, index=events)
            per_variable[case, model] = {m: pd.DataFrame(f[f"per_variable/{m}"][()], index=events,
                                                         columns=settings[case, model]["scored"]) for m in METRICS}
for case in CASES:   # same events and scored measurements for every model of a case
    first = settings[case, MODELS["arx"]]
    for model in MODELS.values():
        assert list(scores[case, model].index) == list(scores[case, MODELS["arx"]].index)
        assert settings[case, model]["scored"] == first["scored"]
        assert settings[case, model]["data_file"] == first["data_file"]
print(f"{len(CASES)} cases x {len(MODELS)} models, {len(events)} test events, "
      f"{len(first['scored'])} scored measurements, left out: {first.get('excluded_variables')}")

8 cases x 3 models, 11 test events, 40 scored measurements, left out: ['xmeas_37']


In [2]:
# Mean ± standard deviation over the test events, per case
tables = {}
for case, description in CASES.items():
    mean = pd.DataFrame({model: scores[case, model].mean() for model in MODELS.values()}).T[METRICS]
    std = pd.DataFrame({model: scores[case, model].std() for model in MODELS.values()}).T[METRICS]
    tables[case] = (mean, std)
    print(description)
    display(mean.round(3).astype(str) + " ± " + std.round(3).astype(str))

No measurement noise, constant disturbances (tep_sp_spaced_6min_clean.h5), set points as covariates


,scaled MAE,MAPE (%),coverage,scaled Winkler
Chronos-2,0.252 ± 0.074,1.04 ± 0.34,0.86 ± 0.08,1.287 ± 0.321
TimesFM 3.0,0.304 ± 0.101,1.257 ± 0.388,0.92 ± 0.048,1.545 ± 0.279
ARX + Ridge,0.184 ± 0.052,0.853 ± 0.212,0.795 ± 0.095,0.939 ± 0.211


Measurement noise and AR(1) disturbances (tep_sp_spaced_6min_noisy.h5), set points as covariates


,scaled MAE,MAPE (%),coverage,scaled Winkler
Chronos-2,0.514 ± 0.061,2.248 ± 0.356,0.812 ± 0.052,2.304 ± 0.238
TimesFM 3.0,0.557 ± 0.067,2.458 ± 0.32,0.83 ± 0.051,2.449 ± 0.195
ARX + Ridge,0.521 ± 0.057,2.353 ± 0.263,0.801 ± 0.054,2.33 ± 0.185


Measurement noise and AR(1) disturbances (tep_sp_spaced_6min_noisy.h5), set points as covariates, disturbances as past-only covariates (ARX cannot use these)


,scaled MAE,MAPE (%),coverage,scaled Winkler
Chronos-2,0.513 ± 0.057,2.251 ± 0.34,0.825 ± 0.046,2.322 ± 0.209
TimesFM 3.0,0.556 ± 0.07,2.456 ± 0.332,0.831 ± 0.053,2.454 ± 0.201
ARX + Ridge,0.521 ± 0.057,2.353 ± 0.263,0.801 ± 0.054,2.33 ± 0.185


... (6 outputs left out)

Measurement noise, constant disturbances (tep_sp_spaced_6min_constdist_noisy.h5), set points and disturbances known over the forecast, errors scaled by the std of tep_sp_spaced_6min_noisy.h5


,scaled MAE,MAPE (%),coverage,scaled Winkler
Chronos-2,0.482 ± 0.051,2.101 ± 0.327,0.825 ± 0.048,2.178 ± 0.208
TimesFM 3.0,0.515 ± 0.058,2.223 ± 0.27,0.849 ± 0.038,2.291 ± 0.135
ARX + Ridge,0.484 ± 0.047,2.145 ± 0.26,0.826 ± 0.042,2.174 ± 0.123


Measurement noise, ramped disturbances (tep_sp_spaced_ramp_noisy.h5), set points and disturbances known over the forecast, errors scaled by the std of tep_sp_spaced_6min_noisy.h5


,scaled MAE,MAPE (%),coverage,scaled Winkler
Chronos-2,0.485 ± 0.053,2.1 ± 0.325,0.811 ± 0.05,2.238 ± 0.324
TimesFM 3.0,0.502 ± 0.061,2.196 ± 0.311,0.849 ± 0.039,2.301 ± 0.206
ARX + Ridge,0.459 ± 0.043,2.015 ± 0.326,0.832 ± 0.036,2.083 ± 0.168


In [3]:
# Cases 7, 2 and 8: same set point changes, different disturbances. Set points as covariates, and the errors of
# every case scaled by the same standard deviation, so the scaled Winkler scores are comparable across the cases
COMBINED_CASES = {"constdist_sp": "7: constant disturbances", "noisy_sp": "2: AR(1) disturbances",
                  "ramp_sp": "8: moving operating point"}
COMBINED_METRICS = ["MAPE (%)", "coverage", "scaled Winkler"]

scale_files = {settings[case, model].get("scale_file", settings[case, model]["data_file"])
               for case in COMBINED_CASES for model in MODELS.values()}
assert len(scale_files) == 1, scale_files
for case in COMBINED_CASES:   # same test events as case 2
    assert list(scores[case, "Chronos-2"].index) == list(scores["noisy_sp", "Chronos-2"].index)
combined = {}
for case, label in COMBINED_CASES.items():
    for model in MODELS.values():
        s = scores[case, model][COMBINED_METRICS]
        combined[label, model] = s.mean().round(3).astype(str) + " ± " + s.std().round(3).astype(str)
combined = pd.DataFrame(combined).T.rename_axis(["case", "model"])
print(f"Mean ± std over the {len(events)} test events, errors scaled by the standard deviation of "
      f"{scale_files.pop()}")
combined

Mean ± std over the 11 test events, errors scaled by the standard deviation of tep_sp_spaced_6min_noisy.h5


MAPE (%)       coverage  \
case                      model                                       
7: constant disturbances  Chronos-2    2.101 ± 0.327  0.825 ± 0.048   
                          TimesFM 3.0   2.223 ± 0.27  0.849 ± 0.038   
                          ARX + Ridge   2.145 ± 0.26  0.826 ± 0.042   
2: AR(1) disturbances     Chronos-2    2.248 ± 0.356  0.812 ± 0.052   
                          TimesFM 3.0   2.458 ± 0.32   0.83 ± 0.051   
                          ARX + Ridge  2.353 ± 0.263  0.801 ± 0.054   
8: moving operating point Chronos-2    2.082 ± 0.305  0.816 ± 0.046   
                          TimesFM 3.0  2.169 ± 0.306    0.85 ± 0.04   
                          ARX + Ridge  2.057 ± 0.349  0.838 ± 0.039   

                                      scaled Winkler  
case                      model                       
7: constant disturbances  Chronos-2    2.178 ± 0.208  
                          TimesFM 3.0  2.291 ± 0.135  
                          ARX + Ridge  2.174 ± 0.123  
2: AR(1) disturbances     Chronos-2    2.304 ± 0.238  
                          TimesFM 3.0  2.449 ± 0.195  
                          ARX + Ridge   2.33 ± 0.185  
8: moving operating point Chronos-2     2.23 ± 0.308  
                          TimesFM 3.0   2.28 ± 0.176  
                          ARX + Ridge  2.117 ± 0.149

In [4]:
# Tables of several runs, all with the errors scaled by the standard deviation of the case 2 data
def run_table(runs, names):
    """Mean ± std over the test events of the runs {(row labels): results file suffix}."""
    scale_files = {settings[case, model].get("scale_file", settings[case, model]["data_file"])
                   for case in runs.values() for model in MODELS.values()}
    assert scale_files == {"tep_sp_spaced_6min_noisy.h5"}, scale_files
    rows = {}
    for labels, case in runs.items():
        assert list(scores[case, "Chronos-2"].index) == list(scores["noisy_sp", "Chronos-2"].index)
        for model in MODELS.values():
            s = scores[case, model][COMBINED_METRICS]
            rows[(*labels, model)] = s.mean().round(3).astype(str) + " ± " + s.std().round(3).astype(str)
    return pd.DataFrame(rows).T.rename_axis([*names, "model"])

In [5]:
# Case 2 data (AR(1) disturbances): what the disturbance signals add, known over the context only or also over the
# forecast. ARX cannot use past-only covariates, so its past-only row equals its set-points-only row
print("Case 2 data, mean ± std over the test events, errors scaled by the std of tep_sp_spaced_6min_noisy.h5")
run_table({("set points only",): "noisy_sp", ("+ disturbances, past only",): "noisy_pastdist",
           ("+ disturbances, past and future",): "noisy_futuredist"}, ["covariates"])

Case 2 data, mean ± std over the test events, errors scaled by the std of tep_sp_spaced_6min_noisy.h5


MAPE (%)       coverage  \
covariates                      model                                       
set points only                 Chronos-2    2.248 ± 0.356  0.812 ± 0.052   
                                TimesFM 3.0   2.458 ± 0.32   0.83 ± 0.051   
                                ARX + Ridge  2.353 ± 0.263  0.801 ± 0.054   
+ disturbances, past only       Chronos-2     2.251 ± 0.34  0.825 ± 0.046   
                                TimesFM 3.0  2.456 ± 0.332  0.831 ± 0.053   
                                ARX + Ridge  2.353 ± 0.263  0.801 ± 0.054   
+ disturbances, past and future Chronos-2     2.24 ± 0.342   0.816 ± 0.05   
                                TimesFM 3.0  2.436 ± 0.348   0.82 ± 0.051   
                                ARX + Ridge  2.263 ± 0.306  0.795 ± 0.054   

                                            scaled Winkler  
covariates                      model                       
set points only                 Chronos-2    2.304 ± 0.238  
                                TimesFM 3.0  2.449 ± 0.195  
                                ARX + Ridge   2.33 ± 0.185  
+ disturbances, past only       Chronos-2    2.322 ± 0.209  
                                TimesFM 3.0  2.454 ± 0.201  
                                ARX + Ridge   2.33 ± 0.185  
+ disturbances, past and future Chronos-2    2.319 ± 0.231  
                                TimesFM 3.0   2.46 ± 0.209  
                                ARX + Ridge  2.298 ± 0.192

In [6]:
# Disturbances known over the context and the forecast on every dataset. With constant disturbances (case 7) the
# disturbance signals never change, so they carry no information
print("Mean ± std over the test events, errors scaled by the std of tep_sp_spaced_6min_noisy.h5")
run_table({(label, covariates): f"{case}_{suffix}"
           for case, label in [("constdist", "7: constant disturbances"), ("noisy", "2: AR(1) disturbances"),
                               ("ramp", "8: moving operating point")]
           for covariates, suffix in [("set points only", "sp"), ("+ disturbances, past and future", "futuredist")]},
          ["case", "covariates"])

Mean ± std over the test events, errors scaled by the std of tep_sp_spaced_6min_noisy.h5


MAPE (%)  \
case                      covariates                      model                        
7: constant disturbances  set points only                 Chronos-2    2.101 ± 0.327   
                                                          TimesFM 3.0   2.223 ± 0.27   
                                                          ARX + Ridge   2.145 ± 0.26   
                          + disturbances, past and future Chronos-2    2.101 ± 0.327   
                                                          TimesFM 3.0   2.223 ± 0.27   
                                                          ARX + Ridge   2.145 ± 0.26   
2: AR(1) disturbances     set points only                 Chronos-2    2.248 ± 0.356   
                                                          TimesFM 3.0   2.458 ± 0.32   
                                                          ARX + Ridge  2.353 ± 0.263   
                          + disturbances, past and future Chronos-2     2.24 ± 0.342   
                                                          TimesFM 3.0  2.436 ± 0.348   
                                                          ARX + Ridge  2.263 ± 0.306   
8: moving operating point set points only                 Chronos-2    2.082 ± 0.305   
                                                          TimesFM 3.0  2.169 ± 0.306   
                                                          ARX + Ridge  2.057 ± 0.349   
                          + disturbances, past and future Chronos-2      2.1 ± 0.325   
                                                          TimesFM 3.0  2.196 ± 0.311   
                                                          ARX + Ridge  2.015 ± 0.326   

                                                                            coverage  \
case                      covariates                      model                        
7: constant disturbances  set points only                 Chronos-2    0.825 ± 0.048   
                                                          TimesFM 3.0  0.849 ± 0.038   
                                                          ARX + Ridge  0.826 ± 0.042   
                          + disturbances, past and future Chronos-2    0.825 ± 0.048   
                                                          TimesFM 3.0  0.849 ± 0.038   
                                                          ARX + Ridge  0.826 ± 0.042   
2: AR(1) disturbances     set points only                 Chronos-2    0.812 ± 0.052   
                                                          TimesFM 3.0   0.83 ± 0.051   
                                                          ARX + Ridge  0.801 ± 0.054   
                          + disturbances, past and future Chronos-2     0.816 ± 0.05   
                                                          TimesFM 3.0   0.82 ± 0.051   
                                                          ARX + Ridge  0.795 ± 0.054   
8: moving operating point set points only                 Chronos-2    0.816 ± 0.046   
                                                          TimesFM 3.0    0.85 ± 0.04   
                                                          ARX + Ridge  0.838 ± 0.039   
                          + disturbances, past and future Chronos-2     0.811 ± 0.05   
                                                          TimesFM 3.0  0.849 ± 0.039   
                                                          ARX + Ridge  0.832 ± 0.036   

                                                                      scaled Winkler  
case                      covariates                      model                       
7: constant disturbances  set points only                 Chronos-2    2.178 ± 0.208  
                                                          TimesFM 3.0  2.291 ± 0.135  
                                                          ARX + Ridge  2.174 ± 0.123  
                          + disturbances, past and future Chronos-2    2.178 ± 0.208  
                                

In [7]:
# Screening: measurements with an inflated percentage error (mean MAPE over the events, per model and case)
LIMIT = 50   # % mean MAPE above which a measurement counts as inflated
mape = pd.DataFrame({(case, model): per_variable[case, model]["MAPE (%)"].mean()
                     for case in CASES for model in MODELS.values()})
worst = mape.max(axis=1).sort_values(ascending=False)
print(f"Largest mean MAPE of any model and case: {worst.iloc[0]:.1f}% ({worst.index[0]}); "
      f"measurements above {LIMIT}%: {list(worst.index[worst > LIMIT]) or 'none'}")
mape.loc[worst.index[:8]].round(1)

Largest mean MAPE of any model and case: 18.9% (xmeas_1); measurements above 50%: none


clean_sp                          noisy_sp                          \
         Chronos-2 TimesFM 3.0 ARX + Ridge Chronos-2 TimesFM 3.0 ARX + Ridge   
xmeas_1        8.9        10.1         5.8      16.5        18.8        18.9   
xmeas_19       7.0         9.6         7.1       7.7         9.9         8.4   
xmeas_39       1.7         2.4         2.4       8.7         8.9         9.1   
xmeas_32       1.5         2.0         1.0       6.9         7.1         6.9   
xmeas_10       4.4         3.2         3.1       6.3         5.9         5.8   
xmeas_38       2.1         3.1         3.0       3.1         3.7         3.8   
xmeas_14       0.3         0.4         0.2       3.4         3.4         3.4   
xmeas_31       1.4         1.9         0.7       2.4         3.0         2.3   

         noisy_pastdist                         constdist_sp  ...     ramp_sp  \
              Chronos-2 TimesFM 3.0 ARX + Ridge    Chronos-2  ... ARX + Ridge   
xmeas_1            16.8        18.8        18.9         14.4  ...        14.9   
xmeas_19            7.9        10.1         8.4          8.0  ...         7.6   
xmeas_39            8.8         8.9         9.1          8.6  ...         8.8   
xmeas_32            6.9         7.1         6.9          6.8  ...         6.8   
xmeas_10            5.9         5.7         5.8          5.4  ...         4.4   
xmeas_38            3.1         3.7         3.8          2.8  ...         3.2   
xmeas_14            3.4         3.4         3.4          3.4  ...         3.4   
xmeas_31            2.4         3.0         2.3          2.2  ...         1.7   

         noisy_futuredist                         constdist_futuredist  \
                Chronos-2 TimesFM 3.0 ARX + Ridge            Chronos-2   
xmeas_1              15.8        17.4        16.3                 14.4   
xmeas_19              8.0        10.1         8.2                  8.0   
xmeas_39              8.8         9.0         9.1                  8.6   
xmeas_32              7.0         7.1         6.9                  6.8   
xmeas_10              6.0         5.8         5.2                  5.4   
xmeas_38              3.1         3.9         3.8                  2.8   
xmeas_14              3.4         3.4         3.4                  3.4   
xmeas_31              2.5         3.0         2.5                  2.2   

                                 ramp_futuredist                          
         TimesFM 3.0 ARX + Ridge       Chronos-2 TimesFM 3.0 ARX + Ridge  
xmeas_1         15.6        15.3            13.5        14.8        13.9  
xmeas_19         9.2         8.5             8.9         9.4         7.5  
xmeas_39         8.7         8.8             8.6         8.9         8.8  
xmeas_32         6.9         6.7             6.9         7.0         6.8  
xmeas_10         4.1         4.7             5.2         4.9         4.3  
xmeas_38         3.4         3.4             3.1         3.3         3.2  
xmeas_14         3.4         3.4             3.4         3.4         3.4  
xmeas_31         2.7         2.3             1.9         2.3         1.6  

[8 rows x 24 columns]